In [11]:
from dotenv import load_dotenv
load_dotenv()

True

##### First PipeLine in RAG 

In [35]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings ,ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


In [13]:
loader = PyPDFLoader("../data/Transformer.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")


Loaded 14 pages


In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

21

In [15]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [9]:
vector_store = Chroma.from_documents(
  documents=splitted_data, 
  embedding=embeddings)

#### Second pipelin in RAG

In [25]:
query = "Why was the Transformer nedded ?"

In [26]:
data = vector_store.similarity_search(query)


In [27]:
context =""
for doc in data:
    context += doc.page_content + "\n"


In [32]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [38]:
def get_context(query:str):
    data = vector_store.similarity_search(query)
    context =""
    for doc in data:
        context += doc.page_content + "\n"


    return {
        "context": context,
        "query": query
    }    
    


In [39]:
prompt = PromptTemplate.from_template(
    """
    Answer the following question based only on the provided context.

    Context:
    {context}

    Question:
    {query}
    """
)

In [40]:
rag_chain = get_context | prompt | llm

In [43]:
res = rag_chain.invoke(query)

print(res.content[0]["text"])

Based on the provided context, the Transformer was needed because previous sequence-based architectures (RNN, LSTM, and GRU) processed words one by one (sequentially), which created limitations when handling long-range relationships between words.

The Transformer solved these issues by using **Self-Attention**, which allows every word to look at every other word simultaneously. This provides several key advantages:

* **Highly parallelizable**
* **Better at understanding context**
* **Effective for long-range relationships**
* **Suitable for large-scale training**
